# PolyPredict - Polymarket Insider Trading Tracker

Este notebook demuestra el sistema completo de detección de insider trading en Polymarket.

## Componentes:
1. **Recolección de datos** - APIs de Polymarket (CLOB, Gamma, Data)
2. **Detección basada en reglas** - Patrones sospechosos inmediatos
3. **Detección ML/DL** - LSTM + Random Forest para análisis avanzado
4. **Visualización y alertas** - Dashboard interactivo

In [ ]:
# Imports
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Our modules
from src.data_collection.polymarket_client import PolymarketClient
from src.detection.rule_based_detector import RuleBasedDetector
from src.models.insider_ml_model import InsiderMLModel

# Visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Imports successful")

## 1. Configuración y Conexión a Polymarket

In [ ]:
# Initialize client
client = PolymarketClient()

# Test connection by fetching active markets
markets = client.get_markets(active=True)

print(f"📊 Connected to Polymarket")
print(f"📈 Found {len(markets)} active markets\n")

# Display top 5 markets
if markets:
    print("Top 5 Active Markets:")
    for i, market in enumerate(markets[:5], 1):
        question = market.get('question', 'N/A')
        volume = market.get('volume', 0)
        print(f"{i}. {question[:60]}... (Volume: ${volume:,.0f})")

## 2. Seleccionar un Market para Analizar

In [ ]:
# Select market with highest volume
if markets:
    selected_market = sorted(
        markets, 
        key=lambda x: x.get('volume', 0), 
        reverse=True
    )[0]
    
    market_id = selected_market.get('condition_id')
    market_question = selected_market.get('question', 'Unknown')
    
    print(f"🎯 Selected Market:")
    print(f"   Question: {market_question}")
    print(f"   Market ID: {market_id}")
    print(f"   Volume: ${selected_market.get('volume', 0):,.0f}")
    print(f"   Active: {selected_market.get('active', False)}")
else:
    print("⚠️  No markets available. Using mock data for demonstration.")
    market_id = "demo_market"
    market_question = "Demo Market"

## 3. Recolectar Datos de Trading

In [ ]:
# Fetch historical trades
print("📥 Fetching historical trading data...")

if market_id and market_id != "demo_market":
    trades_df = client.get_historical_data(market_id, days_back=7)
    
    if not trades_df.empty:
        print(f"✅ Fetched {len(trades_df)} trades")
        print(f"   Date range: {trades_df['timestamp'].min()} to {trades_df['timestamp'].max()}")
        display(trades_df.head())
    else:
        print("⚠️  No trades found. Creating synthetic data...")
        trades_df = None
else:
    trades_df = None

# Create synthetic data if no real data available
if trades_df is None or trades_df.empty:
    print("\n🔧 Generating synthetic trading data for demonstration...")
    
    n_trades = 500
    base_time = datetime.now() - timedelta(days=7)
    
    trades_df = pd.DataFrame({
        'timestamp': [base_time + timedelta(minutes=i*30) for i in range(n_trades)],
        'maker': [f'0x{np.random.randint(0, 100):040x}' for _ in range(n_trades)],
        'size': np.random.exponential(100, n_trades) * np.random.uniform(0.5, 2, n_trades),
        'price': np.random.uniform(0.3, 0.7, n_trades),
        'side': np.random.choice(['buy', 'sell'], n_trades),
        'market': market_id
    })
    
    # Add some suspicious patterns
    # Insider pattern: Large buys 20 minutes before "event"
    event_time = trades_df['timestamp'].max() - timedelta(hours=2)
    suspicious_window = (trades_df['timestamp'] >= event_time - timedelta(minutes=20)) & \
                       (trades_df['timestamp'] < event_time)
    
    # Make some trades suspiciously large
    insider_addresses = [f'0xINSIDER{i:037x}' for i in range(3)]
    for idx in trades_df[suspicious_window].index[:5]:
        trades_df.loc[idx, 'size'] = np.random.uniform(500, 1000)
        trades_df.loc[idx, 'maker'] = np.random.choice(insider_addresses)
        trades_df.loc[idx, 'side'] = 'buy'
    
    print(f"✅ Created {len(trades_df)} synthetic trades with suspicious patterns")
    display(trades_df.head())

## 4. Visualización de Datos de Trading

In [ ]:
# Plot trading activity over time
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Volume over time
trades_df.set_index('timestamp')['size'].resample('1H').sum().plot(
    ax=axes[0], 
    title='Trading Volume Over Time',
    color='steelblue'
)
axes[0].set_ylabel('Volume')
axes[0].grid(True, alpha=0.3)

# Number of trades over time
trades_df.set_index('timestamp')['size'].resample('1H').count().plot(
    ax=axes[1],
    title='Number of Trades Over Time',
    color='coral'
)
axes[1].set_ylabel('Trade Count')
axes[1].grid(True, alpha=0.3)

# Price movement
trades_df.set_index('timestamp')['price'].resample('1H').mean().plot(
    ax=axes[2],
    title='Average Price Over Time',
    color='green'
)
axes[2].set_ylabel('Price')
axes[2].set_xlabel('Time')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Trading Statistics:")
print(f"   Total Volume: ${trades_df['size'].sum():,.2f}")
print(f"   Average Trade Size: ${trades_df['size'].mean():,.2f}")
print(f"   Median Trade Size: ${trades_df['size'].median():,.2f}")
print(f"   Unique Traders: {trades_df['maker'].nunique()}")

## 5. Detección Basada en Reglas

In [ ]:
# Initialize rule-based detector
detector = RuleBasedDetector(
    timing_window_minutes=30,
    volume_spike_threshold=3.0,
    clustering_similarity_threshold=0.7
)

# Set event time (e.g., market resolution or major announcement)
event_time = trades_df['timestamp'].max() - timedelta(hours=2)

print(f"🔍 Running Rule-Based Detection...")
print(f"   Event Time: {event_time}")
print(f"   Timing Window: {detector.timing_window} minutes before event\n")

# Generate detection report
report = detector.generate_report(
    trades_df.copy(),
    market_id,
    event_time
)

print("="*60)
print("📋 DETECTION REPORT")
print("="*60)
print(f"\n{report['summary']}\n")
print(f"Overall Risk Score: {report['risk_score']:.2f}/1.00")
print("="*60)

### 5.1 Timing Anomalies

In [ ]:
timing_anomalies = report['detections']['timing_anomalies']

if timing_anomalies:
    print(f"⚠️  Found {len(timing_anomalies)} timing anomalies:\n")
    
    timing_df = pd.DataFrame(timing_anomalies)
    display(timing_df[[
        'trader', 'minutes_before_event', 'size', 
        'avg_normal_size', 'severity', 'score'
    ]].sort_values('score', ascending=False))
    
    # Visualize
    plt.figure(figsize=(12, 5))
    plt.scatter(
        timing_df['minutes_before_event'],
        timing_df['size'],
        c=timing_df['score'],
        cmap='YlOrRd',
        s=100,
        alpha=0.7
    )
    plt.colorbar(label='Risk Score')
    plt.xlabel('Minutes Before Event')
    plt.ylabel('Trade Size')
    plt.title('Timing Anomalies - Suspicious Pre-Event Trading')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("✅ No timing anomalies detected")

### 5.2 Volume Spikes

In [ ]:
volume_spikes = report['detections']['volume_spikes']

if volume_spikes:
    print(f"📊 Found {len(volume_spikes)} volume spikes:\n")
    
    spikes_df = pd.DataFrame(volume_spikes)
    display(spikes_df[[
        'timestamp', 'volume', 'normal_volume', 
        'spike_ratio', 'severity', 'score'
    ]].sort_values('spike_ratio', ascending=False).head(10))
    
    # Visualize volume spikes
    plt.figure(figsize=(14, 6))
    
    # Plot all volume
    volume_series = trades_df.set_index('timestamp')['size'].resample('10T').sum()
    plt.plot(volume_series.index, volume_series.values, 
             label='Trading Volume', color='steelblue', alpha=0.7)
    
    # Highlight spikes
    spike_times = pd.to_datetime(spikes_df['timestamp'])
    spike_volumes = spikes_df['volume'].values
    plt.scatter(spike_times, spike_volumes, 
                color='red', s=100, label='Volume Spikes', zorder=5)
    
    plt.xlabel('Time')
    plt.ylabel('Volume')
    plt.title('Volume Analysis with Spike Detection')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("✅ No unusual volume spikes detected")

### 5.3 Wallet Clustering

In [ ]:
wallet_clusters = report['detections']['wallet_clusters']

if wallet_clusters:
    print(f"🔗 Found {len(wallet_clusters)} wallet clusters (potential multi-account traders):\n")
    
    for cluster_id, wallets in wallet_clusters.items():
        print(f"\n{cluster_id}: {len(wallets)} wallets")
        for wallet in wallets[:5]:  # Show first 5
            print(f"  - {wallet[:10]}...")
        if len(wallets) > 5:
            print(f"  ... and {len(wallets) - 5} more")
else:
    print("✅ No suspicious wallet clusters detected")

### 5.4 Coordinated Trading

In [ ]:
coordinated = report['detections']['coordinated_trading']

if coordinated:
    print(f"👥 Found {len(coordinated)} coordinated trading events:\n")
    
    coord_df = pd.DataFrame(coordinated)
    display(coord_df[[
        'timestamp', 'num_traders', 'total_volume', 
        'direction', 'severity', 'score'
    ]].sort_values('num_traders', ascending=False))
else:
    print("✅ No coordinated trading patterns detected")

## 6. Machine Learning Detection

In [ ]:
print("🤖 Initializing ML Model...\n")

# Initialize ML model
ml_model = InsiderMLModel(
    sequence_length=50,
    lstm_hidden_size=128
)

# Prepare features from trades
features_df = ml_model.prepare_features(trades_df.copy())

print(f"✅ Extracted {features_df.shape[1]} features from trading data")
print(f"   Features: {list(features_df.columns)}\n")

display(features_df.head())

### 6.1 Crear Datos de Entrenamiento

Para entrenar el modelo, necesitamos datos etiquetados. En un escenario real, estos labels vendrían de:
- Casos confirmados de insider trading
- Análisis histórico de eventos conocidos
- Verificación manual de patrones sospechosos

Por ahora, creamos labels sintéticos basados en las detecciones de reglas.

In [ ]:
# Create synthetic labels based on rule-based detections
# In real scenario, these would come from verified insider trading cases

print("🏷️  Creating training labels...\n")

# Initialize labels (0 = normal, 1 = suspicious)
labels = np.zeros(len(features_df))

# Label based on timing anomalies
if timing_anomalies:
    suspicious_traders = set([t['trader'] for t in timing_anomalies])
    suspicious_mask = trades_df['maker'].isin(suspicious_traders)
    labels[suspicious_mask] = 1

# Label based on volume spikes
if volume_spikes:
    for spike in volume_spikes:
        spike_time = pd.to_datetime(spike['timestamp'])
        time_mask = (trades_df['timestamp'] >= spike_time - timedelta(minutes=5)) & \
                   (trades_df['timestamp'] <= spike_time + timedelta(minutes=5))
        labels[time_mask] = 1

print(f"Label distribution:")
print(f"   Normal trades: {(labels == 0).sum()} ({(labels == 0).mean()*100:.1f}%)")
print(f"   Suspicious trades: {(labels == 1).sum()} ({(labels == 1).mean()*100:.1f}%)")

### 6.2 Entrenar Modelos ML

In [ ]:
from sklearn.model_selection import train_test_split

# Prepare data
X = features_df.values
y = labels

# Create sequences for LSTM
sequences, seq_labels = ml_model.create_sequences(X, y)

print(f"📊 Created {len(sequences)} sequences")
print(f"   Sequence shape: {sequences.shape}")
print(f"   (samples, sequence_length, features)\n")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    sequences, seq_labels, test_size=0.2, random_state=42, stratify=seq_labels
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

In [ ]:
# Train LSTM model
print("\n🧠 Training LSTM Neural Network...\n")

ml_model.train_lstm(
    X_train, y_train,
    X_val=X_test, y_val=y_test,
    epochs=20,
    batch_size=32,
    learning_rate=0.001
)

print("\n✅ LSTM training completed")

In [ ]:
# Train Random Forest
print("\n🌲 Training Random Forest...\n")

ml_model.train_random_forest(
    X_train, y_train,
    n_estimators=100
)

print("\n✅ Random Forest training completed")

### 6.3 Evaluar Modelos

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Make predictions
y_pred_proba = ml_model.predict(X_test, use_ensemble=True)
y_pred = (y_pred_proba > 0.5).astype(int)

print("📊 Model Evaluation:\n")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Suspicious']))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nROC-AUC Score: {roc_auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Suspicious'],
            yticklabels=['Normal', 'Suspicious'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Insider Trading Detection')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

### 6.4 Guardar Modelos Entrenados

In [ ]:
# Save models
ml_model.save_models()

print("💾 Models saved successfully!")
print(f"   Location: {ml_model.model_dir}")

## 7. Análisis de Traders Sospechosos

In [ ]:
# Identify top suspicious traders
print("🔍 Identifying Most Suspicious Traders...\n")

# Get all suspicious trades
all_suspicious = []

for anomaly in timing_anomalies:
    all_suspicious.append({
        'trader': anomaly['trader'],
        'type': 'timing',
        'score': anomaly['score']
    })

# Count by trader
from collections import Counter

trader_scores = {}
for item in all_suspicious:
    trader = item['trader']
    if trader not in trader_scores:
        trader_scores[trader] = []
    trader_scores[trader].append(item['score'])

# Calculate average score per trader
trader_avg_scores = {
    trader: np.mean(scores) 
    for trader, scores in trader_scores.items()
}

# Sort by score
top_suspicious = sorted(
    trader_avg_scores.items(), 
    key=lambda x: x[1], 
    reverse=True
)[:10]

if top_suspicious:
    print("Top 10 Most Suspicious Traders:\n")
    for i, (trader, score) in enumerate(top_suspicious, 1):
        # Get trader's total volume
        trader_volume = trades_df[trades_df['maker'] == trader]['size'].sum()
        num_trades = len(trades_df[trades_df['maker'] == trader])
        
        print(f"{i}. {trader[:15]}...")
        print(f"   Risk Score: {score:.2f}")
        print(f"   Total Volume: ${trader_volume:,.2f}")
        print(f"   Number of Trades: {num_trades}\n")
else:
    print("No suspicious traders identified")

## 8. Dashboard de Resumen

In [ ]:
# Create comprehensive dashboard
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Risk Score Gauge
ax1 = fig.add_subplot(gs[0, 0])
risk_score = report['risk_score']
colors = ['green', 'yellow', 'orange', 'red']
thresholds = [0.25, 0.5, 0.75, 1.0]
color = colors[sum([risk_score > t for t in thresholds])]
ax1.barh([0], [risk_score], color=color)
ax1.set_xlim(0, 1)
ax1.set_title('Overall Risk Score')
ax1.set_xlabel('Risk Level')
ax1.text(risk_score/2, 0, f'{risk_score:.2f}', 
         ha='center', va='center', fontsize=12, fontweight='bold')

# 2. Detection Counts
ax2 = fig.add_subplot(gs[0, 1])
detection_counts = {
    'Timing': len(timing_anomalies),
    'Volume': len(volume_spikes),
    'Coordinated': len(coordinated),
    'Clusters': len(wallet_clusters)
}
ax2.bar(detection_counts.keys(), detection_counts.values(), 
        color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
ax2.set_title('Detections by Type')
ax2.set_ylabel('Count')

# 3. ML Model Performance
ax3 = fig.add_subplot(gs[0, 2])
metrics = ['Precision', 'Recall', 'F1-Score']
# Calculate from classification report
from sklearn.metrics import precision_score, recall_score, f1_score
scores = [
    precision_score(y_test, y_pred, average='binary'),
    recall_score(y_test, y_pred, average='binary'),
    f1_score(y_test, y_pred, average='binary')
]
ax3.bar(metrics, scores, color=['#95E1D3', '#F38181', '#EAFFD0'])
ax3.set_ylim(0, 1)
ax3.set_title('ML Model Performance')
ax3.set_ylabel('Score')

# 4. Volume Timeline
ax4 = fig.add_subplot(gs[1, :])
volume_series = trades_df.set_index('timestamp')['size'].resample('1H').sum()
ax4.plot(volume_series.index, volume_series.values, color='steelblue', linewidth=2)
ax4.set_title('Trading Volume Timeline')
ax4.set_ylabel('Volume')
ax4.grid(True, alpha=0.3)

# 5. Trader Distribution
ax5 = fig.add_subplot(gs[2, 0])
trader_volumes = trades_df.groupby('maker')['size'].sum().sort_values(ascending=False)[:10]
ax5.barh(range(len(trader_volumes)), trader_volumes.values, color='coral')
ax5.set_yticks(range(len(trader_volumes)))
ax5.set_yticklabels([t[:10]+'...' for t in trader_volumes.index])
ax5.set_xlabel('Total Volume')
ax5.set_title('Top 10 Traders by Volume')
ax5.invert_yaxis()

# 6. Price Movement
ax6 = fig.add_subplot(gs[2, 1])
price_series = trades_df.set_index('timestamp')['price'].resample('1H').mean()
ax6.plot(price_series.index, price_series.values, color='green', linewidth=2)
ax6.set_title('Price Movement')
ax6.set_ylabel('Price')
ax6.grid(True, alpha=0.3)

# 7. Suspicious Activity Heatmap
ax7 = fig.add_subplot(gs[2, 2])
# Create hourly heatmap of suspicious activity
trades_df['hour'] = trades_df['timestamp'].dt.hour
trades_df['day'] = trades_df['timestamp'].dt.day
if len(y_pred) == len(trades_df) - ml_model.sequence_length:
    trades_df_subset = trades_df.iloc[ml_model.sequence_length:].copy()
    trades_df_subset['suspicious'] = y_pred
    heatmap_data = trades_df_subset.groupby(['day', 'hour'])['suspicious'].mean().unstack(fill_value=0)
    sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax7, cbar_kws={'label': 'Suspicion Rate'})
    ax7.set_title('Suspicious Activity Heatmap')
    ax7.set_xlabel('Hour of Day')
    ax7.set_ylabel('Day of Month')

plt.suptitle(f'PolyPredict Insider Trading Detection Dashboard\n{market_question}', 
             fontsize=16, fontweight='bold', y=0.995)

plt.show()

print("\n" + "="*80)
print("📊 FINAL SUMMARY")
print("="*80)
print(f"Market: {market_question}")
print(f"Total Trades Analyzed: {len(trades_df)}")
print(f"Overall Risk Score: {risk_score:.2f}/1.00")
print(f"\nDetections:")
print(f"  - Timing Anomalies: {len(timing_anomalies)}")
print(f"  - Volume Spikes: {len(volume_spikes)}")
print(f"  - Coordinated Trading: {len(coordinated)}")
print(f"  - Wallet Clusters: {len(wallet_clusters)}")
print(f"\nML Model Performance:")
print(f"  - ROC-AUC: {roc_auc:.4f}")
print(f"  - Precision: {precision_score(y_test, y_pred, average='binary'):.4f}")
print(f"  - Recall: {recall_score(y_test, y_pred, average='binary'):.4f}")
print("="*80)

## 9. Exportar Resultados

In [ ]:
# Save detection results
import json
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = Path('../data/results')
output_dir.mkdir(parents=True, exist_ok=True)

# Save report as JSON
report_copy = report.copy()
report_copy['analysis_time'] = report_copy['analysis_time'].isoformat()

with open(output_dir / f'detection_report_{timestamp}.json', 'w') as f:
    json.dump(report_copy, f, indent=2, default=str)

# Save suspicious traders
if top_suspicious:
    suspicious_df = pd.DataFrame(top_suspicious, columns=['trader', 'risk_score'])
    suspicious_df.to_csv(output_dir / f'suspicious_traders_{timestamp}.csv', index=False)

print(f"✅ Results exported to {output_dir}")
print(f"   - detection_report_{timestamp}.json")
print(f"   - suspicious_traders_{timestamp}.csv")

## 10. Conclusiones y Próximos Pasos

### Resumen del Sistema

Este notebook ha demostrado un sistema completo de detección de insider trading en Polymarket que combina:

1. **Recolección de Datos en Tiempo Real**
   - APIs de Polymarket (CLOB, Gamma, Data)
   - WebSocket streaming para actualizaciones en vivo

2. **Detección Basada en Reglas**
   - Análisis de timing (operaciones pre-evento)
   - Detección de picos de volumen
   - Clustering de wallets relacionadas
   - Identificación de trading coordinado

3. **Machine Learning Avanzado**
   - LSTM para análisis de series temporales
   - Random Forest para clasificación
   - Ensemble methods para mayor precisión

### Próximos Pasos

Para mejorar el sistema:

1. **Recolectar más datos históricos** para pre-entrenamiento
2. **Validar con casos conocidos** de insider trading
3. **Implementar alertas en tiempo real** vía WebSocket
4. **Agregar análisis on-chain** para rastrear orígenes de fondos
5. **Crear dashboard web** interactivo para monitoreo continuo
6. **Integrar con más fuentes de datos** (noticias, redes sociales, etc.)
7. **Optimizar modelos** con técnicas de AutoML

### Referencias

- Chainalysis wallet clustering analysis
- EPJ Data Science ML insider detection (2024)
- LSTM-based anomaly detection methods
- Random Forest classification for trading patterns